What is done :
- Reteive all financial transaction on HIVE layer 1 blockchain
- Store these transaction in a standardized way on google drive / excel file
- Able to source all outgoing transaction on HIVE L2 (moslty OCLT)

Problem:
- Not able to source incoming transaction on L2 blockchain. Idk why but we can't see on HIVE blockexplored incoming OCLT transaction - see the last transaciton from paulo21 to ocln-finacct, sent 0.2 OCLT, and there is no log on ocln-finacct transactions list.... ?
  - To solve this we need to access L2 API (HIVE engine), no ideas yet how to do it.
  - Something arrise : On L2 transaction (custon JSON), when placing a sell / buy order this generate a transaction - which need to be ignored for the time being


Next steps:
- List all OCL Hive accounts, to gather all transaction
- add a field, in the panda dataframe indicating the name of the source wallet, from where we extract the data (for intercompany transactions)
- Find a way to extract L2 transaction using HIVE engine / VLS or....
- Define acounting scheme per type of transaction to automate accounting entries generation
- source FX rates (idk why but Yahoo finance does not work)
- tbd...

In [15]:
# import all the required libraries
from beem import Hive
from beem.nodelist import NodeList
from beem import Hive
from beem.account import Account
import pandas as pd
from beem.account import Account
from beem.comment import Comment
from beem.exceptions import ContentDoesNotExistsException
from datetime import datetime
import ast
from tqdm import tqdm
import json
from beem.account import Account
import ast
import os

In [ ]:
# get the current list of active nodes
nodelist = NodeList()
nodelist.update_nodes()
nodes = nodelist.get_hive_nodes()
hive = Hive(node=nodes)
print(hive.is_hive)

In [17]:
# Initialize the wallet
hive = Hive()
account = Account("paulo21", blockchain_instance=hive)
hive.wallet.wipe(True)
hive.wallet.unlock("passphrase")

In [18]:
# Def function to convert currency
def convert_currency(_currency):
  if _currency == "@@000000013":
    return "hbd"
  elif _currency == "@@000000021":
    return "hive"
  else:
    return "Error parsing currency"

In [19]:
# Def function to convert json into pandas dataframe
def json_to_dataframe(json):
  df = pd.DataFrame(json)
  return df

In [20]:
# Def function to get all transactions from one account and return a dataframe
def get_all_account_transactions(account_name):
  
  account = Account(account_name, blockchain_instance=hive)

  list_of_transactions_dict = account.history()
  list_of_transactions_dataframe = json_to_dataframe(list_of_transactions_dict)

  return list_of_transactions_dataframe

In [ ]:
# Def a function, which extract all transfer type of transaction for an individual account
def get_transaction_transfer(account_name):

  account = Account(account_name, blockchain_instance=hive)

  total_transactions = len(list(account.history(only_ops=["transfer_from_savings", "transfer_to_savings", "transfer", "claim_reward_balance", "account_create"])))

  transaction_ledger = pd.DataFrame(columns=[    "_date",    "_type",    "_from",    "_memo",    "_to",    "_block",
                                             "_trx_id",    "_id",    "_index",    "_amount_out",    "_amount_in",
                                                 "_currency_out","_currency_in",    "_precision_out",    "_precision_in",
                                                 "_virtual_op", "_key_auths", "_new_account_name"])

#  for trans in account.history():

#    print(trans)


  for trans in tqdm(account.history(only_ops=["transfer_from_savings", "transfer_to_savings", "transfer", "claim_reward_balance",
                                         "account_create"]), total=total_transactions, desc="Processing transactions except collateral"): #two arguments in the only_ops does not work...

    print(trans)

    # Convert date text to date format
    dateString = trans["timestamp"]
    dateformat = "%Y-%m-%dT%H:%M:%S"

    try:
      _date = datetime.strptime(dateString, dateformat)
    except ValueError:
      _date = "Error parsing date format"

    _type = trans["type"]

    if _type in ["transfer_from_savings", "transfer_to_savings", "transfer"]:
      _from = trans["from"]
      _memo = trans["memo"]
      _to = trans["to"]
      _block = trans["block"]
      _trx_id = trans["trx_id"]
      _id = trans["_id"]
      _index = trans["index"]
      _virtual_op = trans["virtual_op"]
      _key_auths = ""
      _new_account_name = ""

      if _from == account_name and _to != account_name :
        _amount_out = int(trans["amount"]["amount"])
        _amount_in = 0
        _currency_out = convert_currency(trans["amount"]["nai"])
        _currency_in = ""
        _precision_out = int(trans["amount"]["precision"])
        _precision_in = 0
        _amount_out = _amount_out / (10**_precision_out)

      elif _from != account_name and _to == account_name :
        _amount_in = int(trans["amount"]["amount"])
        _amount_out = 0
        _currency_in = convert_currency(trans["amount"]["nai"])
        _currency_out = ""
        _precision_in = int(trans["amount"]["precision"])
        _precision_out = 0
        _amount_in = _amount_in / (10**_precision_in)

      elif _from == account_name and _to == account_name :
          _amount_out = int(trans["amount"]["amount"])
          _amount_in = int(trans["amount"]["amount"])
          _currency_out = convert_currency(trans["amount"]["nai"])
          _currency_in = convert_currency(trans["amount"]["nai"])
          _precision_in = int(trans["amount"]["precision"])
          _precision_out = int(trans["amount"]["precision"])
          _amount_in = _amount_in / (10**_precision_in)
          _amount_out = _amount_out / (10**_precision_out)

      transaction_ledger.loc[len(transaction_ledger)] = [_date, _type, _from, _memo, _to, _block, _trx_id, _id,
                                                       _index, _amount_out, _amount_in, _currency_out, _currency_in,
                                                       _precision_out, _precision_in, _virtual_op, _key_auths, _new_account_name ]


    elif _type == "account_create":
      _from = trans["account"]
      _memo = trans["memo_key"]
      _to = ""
      _block = trans["block"]
      _trx_id = trans["trx_id"]
      _id = trans["_id"]
      _index = trans["index"]
      _amount_in = 0
      _currency_out = convert_currency(trans["fee"]["nai"])
      _currency_in = ""
      _precision_out = int(trans["fee"]["precision"])
      _precision_in = 0
      _amount_out = int(trans["fee"]["amount"]) / (10**_precision_out)
      _virtual_op = trans["virtual_op"]
      _key_auths = (trans["active"]["key_auths"])[0]
      _new_account_name = trans["new_account_name"]

      transaction_ledger.loc[len(transaction_ledger)] = [_date, _type, _from, _memo, _to, _block, _trx_id, _id,
                                                       _index, _amount_out, _amount_in, _currency_out, _currency_in,
                                                       _precision_out, _precision_in, _virtual_op, _key_auths, _new_account_name ]

    elif _type == "claim_reward_balance":
      _from = ""
      _memo = ""
      _to = trans["account"]
      _block = trans["block"]
      _trx_id = trans["trx_id"]
      _id = trans["_id"]
      _index = trans["index"]
      _key_auths = ""
      _new_account_name = ""
      _virtual_op = trans["virtual_op"]
      _amount_in_hbd = int(trans["reward_hbd"]["amount"])
      _amount_in_hive = int(trans["reward_hive"]["amount"])
      _currency_out = ""
      _currency_in_hbd = "hbd"
      _currency_in_hive = "hive"
      _precision_out = 0
      _precision_in_hbd = int(trans["reward_hbd"]["precision"])
      _precision_in_hive = int(trans["reward_hive"]["precision"])
      _amount_in_hbd = _amount_in_hbd / (10**_precision_in_hbd)
      _amount_in_hive = _amount_in_hive / (10**_precision_in_hive)
      _amount_out = 0

      if _amount_in_hbd != 0:
        transaction_ledger.loc[len(transaction_ledger)] = [_date, _type, _from, _memo, _to, _block, _trx_id, _id,
                                                       _index, _amount_out, _amount_in_hbd, _currency_out, _currency_in_hbd,
                                                       _precision_out, _precision_in_hbd, _virtual_op, _key_auths, _new_account_name ]
      if _amount_in_hive != 0:
        transaction_ledger.loc[len(transaction_ledger)] = [_date, _type, _from, _memo, _to, _block, _trx_id, _id,
                                                       _index, _amount_out, _amount_in_hive, _currency_out, _currency_in_hive,
                                                       _precision_out, _precision_in_hive, _virtual_op, _key_auths, _new_account_name ]

  return transaction_ledger

wallet_name = "ocln-finacct"
transaction_ledger = get_transaction_transfer(wallet_name)

wallet_name2 = "paulo21"
transaction_ledger2 = get_transaction_transfer(wallet_name2)


In [22]:
def get_json_transaction_transfer(account_name):

  account = Account(account_name, blockchain_instance=hive)

  total_transactions = len(list(account.history(only_ops=["transfer_from_savings", "transfer_to_savings", "transfer", "claim_reward_balance", "account_create"])))

  transaction_ledger = pd.DataFrame(columns=[    "_date",    "_type",    "_from",    "_memo",    "_to",    "_block",
                                             "_trx_id",    "_id",    "_index",    "_amount_out",    "_amount_in",
                                                 "_currency_out","_currency_in",    "_precision_out",    "_precision_in",
                                                 "_virtual_op", "_key_auths", "_new_account_name"])
  

  for trans in tqdm(account.history(only_ops=["custom_json"]), total=total_transactions, desc="Processing Custon_json transactions"): #two arguments in the only_ops does not work...

    # Convert date text to date format
    dateString = trans["timestamp"]
    dateformat = "%Y-%m-%dT%H:%M:%S"

    try:
      _date = datetime.strptime(dateString, dateformat)
    except ValueError:
      _date = "Error parsing date format"

    #Assign type value
    _type = trans["type"]

    #Only filter with ssc-mainnet-hive id
    if trans["id"] == "ssc-mainnet-hive":

      #First we collect the easy parts
      _block = trans["block"]
      _trx_id = trans["trx_id"]
      _virtual_op = trans["virtual_op"]
      _id = trans["_id"]
      _index = trans["index"]
      _key_auths = "Required auth :" + str(trans["required_auths"])








      _from = trans["account"]
      try:
        _memo = json.loads(trans["json"])["contractPayload"]["memo"]
      except Exception as e:
        _memo = "Error parsing memo" + str(e)
      _to = json.loads(trans["json"])["contractPayload"]["to"]
      _currency_out = json.loads(trans["json"])["contractPayload"]["symbol"]
      _amount_out = json.loads(trans["json"])["contractPayload"]["quantity"]
      _amount_in = 0
      
      
      
      
      _currency_in = ""
      _precision_out = 0
      _precision_in = 0
      
      
      _new_account_name = ""

      transaction_ledger.loc[len(transaction_ledger)] = [_date, _type, _from, _memo, _to, _block, _trx_id, _id,
                                                        _index, _amount_out, _amount_in, _currency_out, _currency_in,
                                                        _precision_out, _precision_in, _virtual_op, _key_auths, _new_account_name ]

  return transaction_ledger

    

In [ ]:
# Sort the DataFrame directly without creating a new one
"""transaction_ledger.sort_values(by='_date', inplace=True)
print(transaction_ledger)"""

In [24]:
# Def a function, which extract all transfer type of transaction for an individual account

def get_transaction_collateral(account_name):

  account = Account(account_name, blockchain_instance=hive)

  transaction_ledger = pd.DataFrame(columns=[    "_date",    "_type",    "_from",    "_memo",    "_to",    "_block",
                                             "_trx_id",    "_id",    "_index",    "_amount_out",    "_amount_in",
                                                 "_currency_out","_currency_in",    "_precision_out",    "_precision_in",
                                                 "_virtual_op", "_key_auths", "_new_account_name"])

  total_transactions = len(list(account.history()))

  for trans in tqdm(account.history(), total = total_transactions, desc="Processing collateral transactions"): #two arguments in the only_ops does not work...

    # Convert date text to date format
    dateString = trans["timestamp"]
    dateformat = "%Y-%m-%dT%H:%M:%S"

    try:
      _date = datetime.strptime(dateString, dateformat)

    except ValueError:
      _date = "Error parsing date format"

    _type = trans["type"]

    if _type == "fill_collateralized_convert_request":
      _from = trans["account"]
      _memo = ""
      _to = trans["account"]
      _block = trans["block"]
      _trx_id = trans["trx_id"]
      _id = trans["_id"]
      _index = trans["index"]
      _key_auths = ""
      _new_account_name = ""
      _virtual_op = trans["virtual_op"]

      _amount_in = int(trans["amount_in"]["amount"])
      _currency_in = convert_currency(trans["amount_in"]["nai"])
      _precision_in = int(trans["amount_in"]["precision"])
      _amount_in = _amount_in / (10**_precision_in)

      _amount_out = int(trans["amount_out"]["amount"])
      _currency_out = convert_currency(trans["amount_out"]["nai"])
      _precision_out = int(trans["amount_out"]["precision"])

      _excess_collateral = int(trans["excess_collateral"]["amount"])
      _currency_excess_collateral = convert_currency(trans["excess_collateral"]["nai"])
      _precision_excess_collateral = int(trans["excess_collateral"]["precision"])

      if _currency_in == _currency_excess_collateral :
        _currency_in = _currency_in + _currency_excess_collateral
      else :
        _currency_out = _currency_out + _currency_excess_collateral


      transaction_ledger.loc[len(transaction_ledger)] = [_date, _type, _from, _memo, _to, _block, _trx_id, _id,
                                                       _index, _amount_out, _amount_in, _currency_out, _currency_in,
                                                       _precision_out, _precision_in, _virtual_op, _key_auths, _new_account_name ]


    elif _type == "collateralized_convert":
      _from = ""
      _memo = ""
      _to = trans["owner"]
      _block = trans["block"]
      _trx_id = trans["trx_id"]
      _id = trans["_id"]
      _index = trans["index"]
      _key_auths = ""
      _new_account_name = ""
      _virtual_op = trans["virtual_op"]

      _amount_in = 0
      _currency_in = ""
      _precision_in = ""

      _amount_out = int(trans["amount"]["amount"])
      _currency_out = convert_currency(trans["amount"]["nai"])
      _precision_out = int(trans["amount"]["precision"])

      transaction_ledger.loc[len(transaction_ledger)] = [_date, _type, _from, _memo, _to, _block, _trx_id, _id,
                                                       _index, _amount_out, _amount_in, _currency_out, _currency_in,
                                                       _precision_out, _precision_in, _virtual_op, _key_auths, _new_account_name ]


  return transaction_ledger


In [25]:
def merge_transaction_ledgers(wallet_name):
    """
    Merges two transaction ledgers into a single DataFrame.

    Args:
        transaction_ledger_2: The first transaction ledger (DataFrame).
        transaction_ledger: The second transaction ledger (DataFrame).

    Returns:
        A new DataFrame containing all transactions from both ledgers.
    """
    # Use pd.concat to combine the DataFrames by appending rows

    transaction_ledger_2 = get_transaction_collateral(wallet_name)
    transaction_ledger = get_transaction_transfer(wallet_name)

    merged_ledger = pd.concat([transaction_ledger_2, transaction_ledger], ignore_index=True)
    return merged_ledger

In [ ]:
wallet_name = "ocln-finacct"

transaction_ledger = merge_transaction_ledgers(wallet_name)

#print(transaction_ledger)

In [27]:
# Function : export a pandas DataFrame to an Excel file on a specific folder / path
def export_to_excel(panda_dataframe_data, path_to_folder, file_name):
    """
    Exports a DataFrame to an Excel file.

    Args:
        file_name: The name of the Excel file to create.
        panda_dataframe_data: The DataFrame to export.
    """
    
    # Format the file name
    if not file_name.endswith(".xlsx"):
        file_name += ".xlsx"

    # Create the full file path
    file_path = os.path.join(path_to_folder, file_name)

    # Export the DataFrame to an Excel file
    panda_dataframe_data.to_excel(file_path, index=False)

    print(f"Exported DataFrame to Excel file: {file_name}")


In [ ]:
print(datetime.now())

In [ ]:
# Get from HIVE blockchain all transactions from the account "paulo21"
hive = Hive()
account = Account("paulo21", blockchain_instance=hive)

paulo21_transactions = get_all_account_transactions("paulo21")

file_path = r"C:\Users\triou\iCloudDrive\Personnal Files\Cryptos\Off chain Luxembourg\Accounting\DEV-Python - Data extraction"
file_name = "paulo21_transactions_py" + str(datetime.now()).replace(" ", "_").replace(":","").replace(".","")[:15] + ".xlsx"

export_to_excel(paulo21_transactions, file_path, file_name)